In [ ]:
# Install ultralytics
!pip install ultralytics -q


In [ ]:
# Import libraries
import os
import json as js
from pathlib import Path
import zipfile
import shutil


In [ ]:
# Check dataset structure
data_dir = Path('/kaggle/input/mega-dataset-dental')
print('Dataset directory exists:', data_dir.exists())
if data_dir.exists():
    print('Contents:', list(data_dir.iterdir()))
    # Look for data.yaml
    yaml_path = data_dir / 'data.yaml'
    if yaml_path.exists():
        print('data.yaml found:', yaml_path)
        with open(yaml_path, 'r') as f:
            print(f.read())
    else:
        print('data.yaml NOT found')
    # Check for images and labels
    for split in ['train', 'valid', 'test']:
        img_dir = data_dir / f'{split}/images'
        lbl_dir = data_dir / f'{split}/labels'
        print(f'{split}: images dir exists: {img_dir.exists()}, labels dir exists: {lbl_dir.exists()}')
        if img_dir.exists():
            print(f'  Number of images: {len(list(img_dir.glob("*.*")))}')
        if lbl_dir.exists():
            print(f'  Number of label files: {len(list(lbl_dir.glob("*.txt")))}')


In [ ]:
# If labels are missing, we need to convert COCO annotations
# Check if we have COCO JSON files
coco_dir = data_dir  # assuming JSONs are in root
train_json = coco_dir / 'train_coco.json'
valid_json = coco_dir / 'valid_coco.json'
test_json = coco_dir / 'test_coco.json'
print('COCO train exists:', train_json.exists())
print('COCO valid exists:', valid_json.exists())
print('COCO test exists:', test_json.exists())
# If we have COCO JSONs but no YOLO labels, we convert
def convert_coco_to_yolo(json_path, img_dir, lbl_dir):
    """Convert COCO annotations to YOLO format."""
    lbl_dir.mkdir(parents=True, exist_ok=True)
    with open(json_path, 'r') as f:
        coco = js.load(f)
    # Create mapping from image id to file name and size
    id_to_info = {img['id']: (img['file_name'], img['width'], img['height']) for img in coco['images']}
    # Prepare annotations per image
    ann_per_img = {}
    for ann in coco['annotations']:
        img_id = ann['image_id']
        if img_id not in ann_per_img:
            ann_per_img[img_id] = []
        ann_per_img[img_id].append(ann)
    # Write YOLO label files
    for img_id, anns in ann_per_img.items():
        file_name, width, height = id_to_info[img_id]
        label_file = lbl_dir / (Path(file_name).stem + '.txt')
        with open(label_file, 'w') as f:
            for ann in anns:
                # COCO bbox: [x, y, width, height] (top-left corner)
                x, y, w, h = ann['bbox']
                # YOLO: x_center, y_center, width, height (normalized)
                x_center = (x + w / 2) / width
                y_center = (y + h / 2) / height
                w_norm = w / width
                h_norm = h / height
                # Class ID (assuming category_id starts at 0 or we need to map)
                # In CODO, category_id might be 1-indexed? We'll assume it's already 0-indexed as per names.
                # We'll need to map category_id to class index using categories list.
                # For simplicity, we assume category_id - 1 if starting from 1.
                # We'll load categories to map.
                pass  # We'll implement later
    return
# We'll implement conversion after checking.


In [ ]:
# For now, assume dataset is already in YOLO format.
# We'll create data.yaml if missing.
yaml_path = data_dir / 'data.yaml'
if not yaml_path.exists():
    print('Creating data.yaml')
    yaml_content = '''path: /kaggle/input/mega-dataset-dental
train: train/images
val: valid/images
test: test/images
nc: 5
names: [Caries, Crown, Filling, Implant, Periapical-lesion]'''
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
else:
    print('Using existing data.yaml')
    with open(yaml_path, 'r') as f:
        print(f.read())


In [ ]:
# Verify that we have data to train
train_img_dir = data_dir / 'train/images'
if train_img_dir.exists():
    num_train = len(list(train_img_dir.glob('*.*')))
    print(f'Training images: {num_train}')
else:
    print('WARNING: No training images found. Will attempt to convert COCO.')
    # We'll implement conversion here if needed.
    # For now, we'll raise an error to avoid training on empty data.
    raise FileNotFoundError('No training images found in dataset.')


In [ ]:
# Train YOLOv8m
from ultralytics import YOLO

model = YOLO('yolov8m.pt')  # load pretrained model

# Train the model
results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=1280,
    batch=6,
    mosaic=1.0,
    mixup=0.15,
    cos_lr=True,
    patience=20,  # early stopping after 20 epochs no improvement
    save=True,
    val=True,
    plots=True,
    verbose=True
)

print('Training completed. Best model saved at:', results.best_save_path)


In [ ]:
# Optionally, run validation on test set (if labels exist)
test_img_dir = data_dir / 'test/images'
test_lbl_dir = data_dir / 'test/labels'
if test_img_dir.exists() and test_lbl_dir.exists():
    print('Running validation on test set...')
    metrics = model.val(data=str(yaml_path), split='test')
    print('Test metrics:', metrics)
else:
    print('Test labels not found, skipping test validation.')


In [ ]:
# Save the best model to output (will be available in /kaggle/working)
import shutil
best_model = Path(results.best_save_path)
output_dir = Path('/kaggle/working')
shutil.copy(best_model, output_dir / 'best.pt')
print('Model copied to /kaggle/working/best.pt')
